# 01 - Data Loading and Inspection
**Dataset:** Telco Customer Churn (IBM sample data, via Kaggle)
**Goal:** Understand the structure of the data and find quality issues before modeling.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
print("shape:" , df.shape)
df.info()

shape: (7043, 21)
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   s

In [3]:
df.describe()
df.describe(include='object')

C:\Users\RAJ KUMAR\AppData\Local\Temp\ipykernel_1380\754461004.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object')


,customerID,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,TotalCharges,Churn
count,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043
unique,7043,2,2,2,2,3,3,3,3,3,3,3,3,3,2,4,6531,2
top,7590-VHVEG,Male,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,20.2,No
freq,1,3555,3641,4933,6361,3390,3096,3498,3088,3095,3473,2810,2785,3875,4171,2365,11,5174


In [4]:
print(df.isnull().sum())
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).round(3))

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


In [5]:
bad = df[pd.to_numeric(df['TotalCharges'], errors='coerce').isnull()]
print (df['TotalCharges'].dtype)
print(df ['TotalCharges'].isnull().sum())

str
0


## Findings
- 7,043 customers, 21 columns.
- `isnull()` showed no missing values, but `TotalCharges` was stored as text.
- 11 rows had blank spaces instead of numbers; all are customers with tenure = 0.
- Target `Churn` is imbalanced: about 73% No / 27% Yes.

# Part 2 - Cleaning
Decisions and reasons are noted for each step.

In [6]:
print ("Before Cleaning:" , df.shape)
df_raw = df.copy()

Before Cleaning: (7043, 21)


In [7]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(df['TotalCharges'].dtype)
print(df['TotalCharges'].isnull().sum())

float64
11


In [8]:
missing = df[df['TotalCharges'].isnull()]
print(len(missing))
print(missing['tenure'].unique())

11
[0]


In [9]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)
print(df['TotalCharges'].isnull().sum())

0


**TotalCharges:** 11 blanks, all with tenure = 0 (new customers, not yet billed). Filled with 0 because no charges have accrued.

In [10]:
print(df['customerID'].nunique(), "unique IDs out of", len(df))
df = df.drop('customerID', axis=1)
print(df.shape)

7043 unique IDs out of 7043
(7043, 20)


In [11]:
print(df.duplicated().sum())

22


**Duplicates:** A few rows are identical after removing customerID. Since there is no unique key, they are most likely different customers with the same plan, so they are kept.

In [12]:
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
for c in service_cols + ['MultipleLines']:
    print(c, df[c].unique())

OnlineSecurity <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
OnlineBackup <StringArray>
['Yes', 'No', 'No internet service']
Length: 3, dtype: str
DeviceProtection <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
TechSupport <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
StreamingTV <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
StreamingMovies <StringArray>
['No', 'Yes', 'No internet service']
Length: 3, dtype: str
MultipleLines <StringArray>
['No phone service', 'No', 'Yes']
Length: 3, dtype: str


In [13]:
for c in service_cols:
    df[c] = df[c].replace('No internet service', 'No')
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

for c in service_cols + ['MultipleLines']:
    print(c, df[c].unique())

OnlineSecurity <StringArray>
['No', 'Yes']
Length: 2, dtype: str
OnlineBackup <StringArray>
['Yes', 'No']
Length: 2, dtype: str
DeviceProtection <StringArray>
['No', 'Yes']
Length: 2, dtype: str
TechSupport <StringArray>
['No', 'Yes']
Length: 2, dtype: str
StreamingTV <StringArray>
['No', 'Yes']
Length: 2, dtype: str
StreamingMovies <StringArray>
['No', 'Yes']
Length: 2, dtype: str
MultipleLines <StringArray>
['No', 'Yes']
Length: 2, dtype: str


In [14]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(df['Churn'].value_counts())
print(df['Churn'].isnull().sum())

Churn
0    5174
1    1869
Name: count, dtype: int64
0


In [15]:
df.groupby('Contract')['Churn'].mean().round(3)

Contract
Month-to-month    0.427
One year          0.113
Two year          0.028
Name: Churn, dtype: float64

In [16]:
print("Shape:", df.shape)
print("Total missing:", df.isnull().sum().sum())
print(df.dtypes.value_counts())

assert df.isnull().sum().sum() == 0, "Missing values remain"
assert set(df['Churn'].unique()) == {0, 1}, "Target not encoded correctly"
assert 'customerID' not in df.columns
print("All checks passed")

Shape: (7043, 20)
Total missing: 0
str        15
int64       3
float64     2
Name: count, dtype: int64
All checks passed


In [17]:
df.to_csv("../data/cleaned_churn.csv", index=False)

In [18]:
check = pd.read_csv("../data/cleaned_churn.csv")
print(check.shape, check.isnull().sum().sum())

(7043, 20) 0


## Cleaning Summary
| Step | Issue | Action | Reason |
|---|---|---|---|
| 1 | `TotalCharges` stored as text, 11 blanks | Converted to numeric; filled blanks with 0 | All 11 are new customers (tenure = 0), so nothing billed yet |
| 2 | `customerID` | Dropped | Unique identifier, no predictive value |
| 3 | Duplicate rows | Kept | No unique key after dropping ID; likely different customers with the same plan |
| 4 | "No internet service" / "No phone service" | Replaced with "No" | Information already captured in `InternetService` and `PhoneService` |
| 5 | `Churn` as Yes/No | Encoded to 1/0 | Models need numeric targets |

**Result:** 7,043 rows, 20 columns, no missing values. Saved to `data/cleaned_churn.csv`.